# The goal of this notebook is to see if the OAMFA manges to find any signals "through" the clouds

This should mainly test if \mu and \Psi maps to physical phenomenoms, preferably the tru spectral signature of a material, and sensor noise.

#### **Data**
Two images: 
1) HYPSO-1_HSI_20231004T104030Z-l1a.nc 
2) HYPSO-1_HSI_20231004T104030Z-l1a.nc

Training-set: The whole images.\
Test-set: One pixel from each of the distinct biases (water, land, alge water).\

#### **Tests**
Stream the image containing clouds


#### **Plot**
Plot the different cluster asignments to see what materials the different clusters represetns
Plot the compoent-heatmap for each pixel, see if you can "see" water and land throug the clouds!




In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch

from matplotlib.colors import ListedColormap
import matplotlib.patches as patches

parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from otfp import MFA_OTFP
from hypso import Hypso


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

L2_NORMALIZATION = True  # Set to True to enable L2 normalization, False to disable

In [ ]:

# 0. FILE PATHS & CONFIG
cloud_image_path = "../data/cloud_test/cloud_2025-03-11T09-39-18Z-l1b.nc" 


# 1. LOAD DATA 
print(f"Loading baseline hypercube from {cloud_image_path}...")
try:
    satobj = Hypso(cloud_image_path)
    cube = getattr(satobj, "l1b_cube", None)
    if cube is None:
        raise ValueError("Missing 'l1b_cube' data.")
    img_full = cube.values.astype(np.float32)
    print(f"Successfully loaded | Shape: {img_full.shape}")
except Exception as e:
    print(f"FATAL: Error processing {cloud_image_path}: {e}")
    sys.exit()


# 4. PLOTTING THE SETUP
# --- PRE-COMPUTE RGB IMAGE (Just like the second plot) ---
r_band, g_band, b_band = 70, 50, 30  
rgb_cube = img_full[:, :, [r_band, g_band, b_band]]

# Normalize the RGB image to [0, 1] using percentiles
p2, p98 = np.percentile(rgb_cube, (2, 98))
rgb_normalized = np.clip((rgb_cube - p2) / (p98 - p2 + 1e-8), 0, 1)


# Create a wide horizontal figure 
fig, axes = plt.subplots(1, 1, figsize=(14, 10)) 

# Plot with aspect='auto' to stretch the image correctly
axes.imshow(rgb_normalized, aspect='auto')


# Update labels to match the transposed orientation
axes.set_xlabel("Flight Path / Time (Push-broom Row Index)", fontsize=14)
axes.set_ylabel("Pixel Width", fontsize=14)

# Helper to place markers on the Transposed image 

plt.tight_layout()
plt.show()

### Cloud distributions

In [ ]:
def plot_selected_spectra(cube, rgb_image, points, wavelengths=None):

    num_points = len(points)
    # Generate distinct colors for each point
    colors = plt.cm.tab10(np.linspace(0, 1, max(10, num_points)))
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    
    # --- Plot 1: The RGB Image with Markers ---
    ax1 = axes[0]
    ax1.imshow(rgb_image, aspect='auto')
    ax1.set_title("Sample Locations", fontsize=14, fontweight='bold')
    ax1.set_ylabel("Pixel Width (Spatial)")
    ax1.set_xlabel("Flight Path (Push-broom Row Index)")
    
    # --- Plot 2: The Spectral Signatures ---
    ax2 = axes[1]
    ax2.set_title("Spectral Signatures", fontsize=14, fontweight='bold')
    ax2.set_xlabel("Band Index" if wavelengths is None else "Wavelength (nm)", fontsize=12)
    ax2.set_ylabel("Radiance / Reflectance", fontsize=12)
    ax2.grid(True, linestyle='--', alpha=0.5)
    
    x_axis = wavelengths if wavelengths is not None else np.arange(cube.shape[2])
    
    # Loop through the points, plot markers, and plot spectra
    for i, (row, col) in enumerate(points):
        color = colors[i]
        label_text = f"P{i+1} ({row}, {col})"
        
        # Mark the location on the RGB image
        ax1.plot(col, row, marker='o', markersize=8, markeredgecolor='white', 
                 markerfacecolor=color, linestyle='None', label=label_text)
        
        # Extract and plot the 1D spectrum for that specific pixel
        spectrum = cube[row, col, :]
        ax2.plot(x_axis, spectrum, color=color, linewidth=2, label=label_text)
        
    ax1.legend(loc="upper right")
    ax2.legend(loc="upper right")
    
    plt.tight_layout()
    plt.show()

# --- HOW TO USE IT ---
# Define a list of (row, col) coordinates you want to inspect.
# Adjust these based on the actual dimensions of your img_test!
target_pixels = [
    (550, 95),   # Try to pick a clear water pixel
    (270, 780),  # Try to pick a thick cloud pixel
    (450, 400),   # Try to pick a thin cloud/edge pixel
    (220, 1020)   # Try to pick a clear land pixel
]

# Call the function (assuming img_test and rgb_normalized are still in memory)
plot_selected_spectra(img_full, rgb_normalized, target_pixels)

### Training only on clean pixels

In [ ]:
# 1. Extract clean patches of BOTH materials
clean_water_patch = img_full[170:190, 600:700, :]   # Water patch
clean_land_patch = img_full[450:470, 900:1000, :]   # Land patch

d = img_full.shape[2]
water_flat = clean_water_patch.reshape(-1, d)
land_flat = clean_land_patch.reshape(-1, d)

water_tensor = torch.tensor(water_flat, dtype=torch.float32).to(device)
land_tensor = torch.tensor(land_flat, dtype=torch.float32).to(device)

# 2. Initialize the Model
MFA_OTFP_cloud_model = MFA_OTFP(
    n_channels=d, 
    device=device, 
    outlier_update_treshold=100,
    q_max=4,
    L2_normalization=L2_NORMALIZATION
)

# 3. FORCE COMPONENT 0: Fit exclusively on Water
print(f"Training Component 0 on {water_flat.shape[0]} Water pixels...")
MFA_OTFP_cloud_model.fit(water_tensor)
print(f"K after Water: {MFA_OTFP_cloud_model.MFA.K}")

# 4. FORCE COMPONENT 1: Stream Land to trigger an Outlier Spawn
print(f"Streaming {land_flat.shape[0]} Land pixels to spawn Component 1...")
with torch.no_grad():
    # stepwise_updates=True allows the model to learn the Land as a new class
    MFA_OTFP_cloud_model.process_data_block(X=land_tensor, stepwise_updates=True)

final_k = MFA_OTFP_cloud_model.MFA.K
print(f"K after Land streaming: {final_k} (Baseline model ready!)")

# =====================================================================
# 5. Evaluate and Plot the Assignment Map to verify the split
# =====================================================================

# Combine them just for the visual plot
clean_train_data = np.concatenate([clean_water_patch, clean_land_patch], axis=0)
clean_train_flat = clean_train_data.reshape(-1, d)
clean_train_tensor = torch.tensor(clean_train_flat, dtype=torch.float32).to(device)

with torch.no_grad():
    # Use your wrapper safely; stepwise_updates=False to freeze it
    clean_assignments, _, _ = MFA_OTFP_cloud_model.process_data_block(
        X=clean_train_tensor, 
        stepwise_updates=False
    )
    
assignments_np = clean_assignments.cpu().numpy().reshape(clean_train_data.shape[0], clean_train_data.shape[1])

cmap_classes = ListedColormap(plt.cm.tab10.colors[:final_k]) 
plt.figure(figsize=(8, 6))

plt.imshow(assignments_np, cmap=cmap_classes, aspect='auto', vmin=-0.5, vmax=final_k-0.5)
plt.title("Component Assignments on Clean Training Area", fontsize=14)
plt.colorbar(ticks=range(final_k), label="Component ID")
plt.xlabel("Column Index")
plt.ylabel("Row Index")


In [ ]:
r_band, g_band, b_band = 70, 50, 30  
rgb_cube = img_full[:, :, [r_band, g_band, b_band]]

# Normalize the RGB image to [0, 1] using percentiles
p2, p98 = np.percentile(rgb_cube, (2, 98))
rgb_normalized = np.clip((rgb_cube - p2) / (p98 - p2 + 1e-8), 0, 1)


# Create a wide horizontal figure 
fig, axes = plt.subplots(1, 1, figsize=(14, 6.5)) 

# Plot with aspect='auto' to stretch the image correctly
axes.imshow(rgb_normalized, aspect='auto')

# Update labels to match the transposed orientation
axes.set_xlabel("Flight Path / Time (Push-broom Row Index)", fontsize=14)
axes.set_ylabel("Pixel Width", fontsize=14)

# Helper to place markers on the Transposed image 

plt.tight_layout()
plt.show()


h_full, w_full, _ = img_full.shape

# 1. Prepare the entire image as a single tensor
full_tensor = torch.tensor(img_full.reshape(-1, d), dtype=torch.float32).to(device)

print("Running full image inference (Model Frozen)...")
with torch.no_grad():
    # FIX: Apply L2 Normalization before evaluating!
    if MFA_OTFP_cloud_model.L2_normalization:
        full_tensor = torch.nn.functional.normalize(full_tensor, p=2, dim=1)
        
    # Unpack log_probs instead of just log_resp_norm
    log_resp_norm, _, log_probs, mahalanobis_dists = MFA_OTFP_cloud_model.MFA.e_step(X=full_tensor)
    assignments = torch.argmax(log_resp_norm, dim=1)

# 3. Reshape the absolute Log-Likelihoods (NOT the normalized probabilities)
# We don't take torch.exp() because likelihoods of continuous distributions can be > 1 or very small.
# The raw log_probs will give us a perfect gradient map!
log_probs_np = log_probs.cpu().numpy()
likelihood_map = log_probs_np.reshape(h_full, w_full, final_k)

print("Inference complete. Plotting Absolute Likelihood Heatmaps...")

# 4. Create a dedicated plot for the Absolute Likelihoods
if final_k > 0:
    fig, axes = plt.subplots(final_k, 1, figsize=(14, 5 * final_k), sharex=True)
    if final_k == 1: axes = [axes]
        
    for k in range(final_k):
        ax = axes[k]
        
        # Extract the 2D Absolute Log-Likelihood map for this component
        likelihood_2d = likelihood_map[:, :, k]
        
        # Plot using an un-bounded color scale (vmin/vmax removed)
        # Use robust percentiles to stretch the contrast nicely
        p5, p95 = np.percentile(likelihood_2d, (5, 95))
        im = ax.imshow(likelihood_2d, cmap='magma', aspect='auto', vmin=p5, vmax=p95)
        
        ax.set_title(f"Absolute Log-Likelihood Map: Component {k} (Land/Water)", fontsize=12, fontweight='bold')
        ax.set_ylabel("Pixel Width")
        
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Log( P(x|k) )', rotation=270, labelpad=15)

    axes[-1].set_xlabel("Flight Path / Time (Push-broom Row Index)", fontsize=12)
    plt.tight_layout()
    plt.show()

# plotting the true imahe with no clouds from the file "../data/cloud_test/noclouds_2023-10-04T10-40-30Z-l1b.nc" 

no_cloud_image_path = "../data/cloud_test/nocloud_2024-06-15T10-48-13Z-l1b.nc"  

# 1. LOAD DATA 
print(f"Loading baseline hypercube from {no_cloud_image_path}...")
try:
    satobj = Hypso(no_cloud_image_path)
    cube = getattr(satobj, "l1b_cube", None)
    if cube is None:
        raise ValueError("Missing 'l1b_cube' data.")
    img_full_no_clouds = cube.values.astype(np.float32)
    print(f"Successfully loaded | Shape: {img_full_no_clouds.shape}")
except Exception as e:
    print(f"FATAL: Error processing {no_cloud_image_path}: {e}")
    sys.exit()

r_band, g_band, b_band = 70, 50, 30  
rgb_cube = img_full_no_clouds[:, :, [r_band, g_band, b_band]]

# Normalize the RGB image to [0, 1] using percentiles
p2, p98 = np.percentile(rgb_cube, (2, 98))
rgb_normalized = np.clip((rgb_cube - p2) / (p98 - p2 + 1e-8), 0, 1)


# Create a wide horizontal figure 
fig, axes = plt.subplots(1, 1, figsize=(14, 5)) 

# Plot with aspect='auto' to stretch the image correctly
axes.imshow(rgb_normalized, aspect='auto')

# Update labels to match the transposed orientation
axes.set_xlabel("Flight Path / Time (Push-broom Row Index)", fontsize=14)
axes.set_ylabel("Pixel Width", fontsize=14)

# Helper to place markers on the Transposed image 

plt.tight_layout()
plt.show()